# **5G Deployment Example With RAN550 Benetel O-RAN 7.2:**

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UDM, UDR, UPF
- OAI split RAN: CU-CP, CU-UP and two DUs with RAN550 Benetel RUs in R2Lab (`benetel1` and `benetel2`)
- COTS UEs: `qhat01`, `qhat02` and `qhat03` in R2Lab

The `ifran` library supports complex experimentation workflows through its flexible, programmatic features.

This example puts many of those features to work:

- **Fine-grained customization:** tuning the 5G deployment extensively through advanced parameters in the `metamodel` and `deployment_model`.
- **Programmatic model generation:** building both models in code rather than writing them by hand.
- **Custom playbooks:** running them with the `MiscellaneousManagement` class.
- **Custom data:** manipulating and using it through the `miscellaneous` field in the `deployment_model`.
- **Customized component deployments:** adapting 5G components with `custom_chart_path` and `custom_rendered_files`.

It also touches on several other capabilities along the way.

## Install Dependencies:

In [ ]:
! pip install ifran

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

In [ ]:
! pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

In [ ]:
! slices auth login

## Input 5G Network Description:

### Metamodel:

In [1]:
def make_ru_meta() -> dict:
    return {
        "type": "real",
        "timeSource": "external",
        "clockSource": "external",
        "numberOfTxPorts": 4,
        "numberOfRxPorts": 4,
        "txAttenuationDB": 0,
        "rxAttenuationDB": 0,
        "rxGainDB": 75,
    }

def make_du_meta(du_id, point_a_arfcn, ssb_arfcn, ru_name, physical_cell_id) -> dict:
    return {
        "type": "du",
        "config": {
            "radioConfiguration": {
                "nrBand": 78,
                "channelBandwidthMhz": 100,
                "pointAFrequencyArfcn": point_a_arfcn,
                "ssbArfcn": ssb_arfcn,
                "ssbPbchBlockPowerDbm": -15,
                "prachConfigurationIndex": 152,
                "msg1FrequencyStartPrb": 0,
                "tddConfiguration": {
                    "periodicityMilliseconds": 2.5,
                    "nrofDownlinkSlots": 3,
                    "nrofDownlinkSymbols": 6,
                    "nrofUplinkSlots": 1,
                    "nrofUplinkSymbols": 4,
                },
            },
            "gnbDuId": du_id,
            "nrCellId": du_id,
            "physicalCellId": physical_cell_id,
            "ru": ru_name,
            "cu_cp": "cucp",
            "cu_up": "cuup",
        },
    }

def make_ue_meta(supi, gnbs) -> dict:
    return {
        "type": "cots",
        "config": {
            "supi": supi,
            "authenticationData": {
                "k": "fec86ba6eb707ed08905757b1bb44b8f",
                "opc": "C42449363BBAD02B66D16BC975D77CC1",
            },
            "gnbs": gnbs,
        },
    }

In [2]:
from ifran import MetaModel

metamodel = MetaModel.model_validate(
{
  "rus": {
      "benetel1": make_ru_meta(),
      "benetel2": make_ru_meta(),
  },
  "ran": {
      "components": {
          "cucp": {"type": "cu_cp"},
          "cuup": {
              "type": "cu_up",
              "config": {"cu_cp": "cucp"},
          },
          "du1": make_du_meta(
              du_id=1,
              point_a_arfcn=626772,
              ssb_arfcn=630048,
              ru_name="benetel1",
              physical_cell_id=0,
          ),
          "du2": make_du_meta(
              du_id=2,
              point_a_arfcn=646644,
              ssb_arfcn=649920,
              ru_name="benetel2",
              physical_cell_id=1,
          ),
      }
  },
  "ues": {
      f"qhat{i+1:02d}": make_ue_meta(supi=f"00101000000000{6 + i}", gnbs=["du1"]) for i in range(3)
  },
})

### Deployment model and use-case parameters

In [3]:
! pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [4]:
import os

from dotenv import load_dotenv

load_dotenv()

R2LAB_SLICE = os.getenv("R2LAB_SLICE")
R2LAB_EMAIL = os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD = os.getenv("R2LAB_PASSWORD")

SLICES_PROJECT = os.getenv("SLICES_PROJECT")
SLICES_EXPERIMENT = os.getenv("SLICES_EXPERIMENT")
SLICES_BI_INFRA = os.getenv("SLICES_BI_INFRA")
SLICES_BI_VM_FLAVOR = os.getenv("SLICES_BI_VM_FLAVOR")
SLICES_BI_VM_DURATION = os.getenv("SLICES_BI_VM_DURATION")

In [5]:
DU_RESOURCES = {
    "define": True,
    "requests": {"nf": {"cpu": 14, "memory": "8Gi"}},
    "limits": {"nf": {"cpu": 14, "memory": "8Gi"}},
}

DU_ADVANCED_CONFIG = {
    "local_rf": "no",
    "useAdditionalOptions": "--log_config.global_log_options level,nocolor,time",
    "tr_preference": "raw_if4p5",
    "sl_ahead": 5,
    "pdsch_AntennaPorts_XP": 2,
    "pdsch_AntennaPorts_N1": 2,
    "pusch_AntennaPorts": 4,
    "do_CSIRS": 1,
    "do_SRS": "periodic",
    "max_pdschReferenceSignalPower": -27,
    "ssb_PositionsInBurst_Bitmap": 1,
    "initialDLBWPlocationAndBandwidth": 1099,
    "initialDLBWPcontrolResourceSetZero": 11,
    "initialDLBWPsearchSpaceZero": 0,
    "initialULBWPlocationAndBandwidth": 1099,
    "prach_msg1_FDM": 0,
    "zeroCorrelationZoneConfig": 0,
    "preambleReceivedTargetPower": -100,
    "ssb_perRACH_OccasionAndCB_PreamblesPerSSB_PR": 4,
    "ssb_perRACH_OccasionAndCB_PreamblesPerSSB": 14,
    "p0_NominalWithGrant": -96,
    "p0_nominal": -96,
    "pMax": 23,
    "preambleTransMax": 8,
    "powerRampingStep": 3,
    "msg3_DeltaPreamble": 2,
    "hoppingId": 0,
    "pusch_TargetSNRx10": 150,
    "pucch_TargetSNRx10": 220,
    "pusch_RSSI_Threshold": -150,
    "pucch_RSSI_Threshold": -150,
    "dl_bler_target_lower": 0.15,
    "dl_bler_target_upper": 0.35,
    "ul_bler_target_lower": 0.15,
    "ul_bler_target_upper": 0.35,
    "pusch_FailureThres": 500,
    "ul_max_mcs": 28,
    "stats_max_ue": 32,
    "prach_dtx_threshold": 140,
    "pucch0_dtx_threshold": 10,
    "pusch_dtx_threshold": 10,
    "max_ldpc_iterations": 15,
    "tx_amp_backoff_dB": 12,
    "phase_compensation": 0,
    "fhi_72": {
        "dpdk_mem_size": 0,
        "du_addr": ["00:11:22:33:44:67", "00:11:22:33:44:66"],
        "ru_addr": ["70:b3:d5:e1:5b:81", "70:b3:d5:e1:5b:81"],
        "mtu": 9600,
        "file_prefix": "fhi_72",
        "fh_config": [
            {
                "T1a_cp_dl": [419, 470],
                "T1a_cp_ul": [285, 336],
                "T1a_up": [294, 345],
                "Ta4": [0, 200],
                "ru_config": {
                    "iq_width": 9,
                    "iq_width_prach": 9,
                },
            }
        ],
    },
}

POS_BOOT_PARAMS = "net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63 nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll selinux=0 enforcing=0 intel_pstate=disable nosoftlockup hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt"

VMS = {
    f"node{i}": {"provisioner": "bi"}
    for i in range(1, 5)
}

BAREMETAL_NODES = {
    f"{node}": {
        "provisioner": "pos",
        "roles": ["ran", "du"],
        "k8s_params": {
            "kubelet_params": {
                "cpu_manager_policy": "static",
                "cpu_manager_policy_options": {"full-pcpus-only": "true"},
                "topology_manager_policy": "best-effort",
            },
        },
    }
    for node in ["sopnode-f3", "sopnode-f4"]
}

In [6]:
def make_du_deployment(node_name: str) -> dict:
    return {
        "type": "du",
        "node_name": node_name,
        "resources": DU_RESOURCES,
        "advanced_config": DU_ADVANCED_CONFIG,
        "custom_chart_path": "./custom_du_chart/oai-du",
        "custom_rendered_files": ["./custom_du_chart/values.yaml.j2"],
    }


def make_ru_deployment() -> dict:
    return {
        "vendor": "benetel",
        "model": "ran550",
        "provisioner": "r2lab",
        "ssh_info": {
            "extra_ansible_vars": {
                "ansible_python_interpreter": "", # Benetel RUs don't have Python
            },
        },
    }


def make_ue_deployment() -> dict:
    return {
        "type": "cots",
        "provisioner": "r2lab",
    }

## Local FHI72 and Benetel models

In [7]:
from typing import Literal, Self

from metamodel_3gpp.models.base import StrictBaseModel
from pydantic import Field, model_validator


class Fhi72VF(StrictBaseModel):
    """One SR-IOV virtual function used by an O-RAN 7.2 fronthaul plane."""

    resource_name: str = Field(pattern=r"^[A-Za-z0-9_]+$")
    plane: Literal["U", "C"]
    mac: str = Field(pattern=r"^([0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}$")
    vlan: int = Field(ge=1, le=4094)


class Fhi72PTPParams(StrictBaseModel):
    """PTP settings for the node's linuxptp services."""

    interface: str
    domain_number: int = Field(default=24, ge=0, le=255)
    network_transport: Literal["L2", "UDPv4", "UDPv6"] = "L2"
    log_announce_interval: int = 1
    log_sync_interval: int = -4
    linuxptp_version: str = "v4.4"


class Fhi72NodeParams(StrictBaseModel):
    """O-RAN 7.2 node settings consumed by the example Ansible roles."""

    fronthaul_interface: str
    mtu: int = Field(default=9216, ge=1500, le=9728)
    vfs: list[Fhi72VF] = Field(min_length=2)
    resource_prefix: str = "openshift.io"
    hugepages_1g: int = Field(default=0, ge=0)
    ring_size: int | None = Field(default=8160, ge=64)
    reserved_cpus: str = "0-5"
    topology_manager_policy: Literal["none", "best-effort", "restricted", "single-numa-node"] = "best-effort"
    irq_pattern: str | None = None
    ptp: Fhi72PTPParams | None = None
    device_plugin_image: str = "ghcr.io/k8snetworkplumbingwg/sriov-network-device-plugin:v3.9.0"

    @model_validator(mode="after")
    def validate_planes(self) -> Self:
        planes = sorted(vf.plane for vf in self.vfs)
        if planes != ["C", "U"]:
            raise ValueError("Configure exactly one C-plane VF and one U-plane VF.")
        if len({vf.vlan for vf in self.vfs}) != 1:
            raise ValueError("The C-plane and U-plane VFs must use the same VLAN.")
        return self

    def vf(self, plane: Literal["U", "C"]) -> Fhi72VF:
        return next(vf for vf in self.vfs if vf.plane == plane)


class BenetelRUParams(StrictBaseModel):
    """Benetel RAN550 settings combined with the metamodel radio configuration."""

    tx_power_dbm: float = 24.0
    downlink_scaling: Literal[0, 6, 12, 18] = 0
    compression: Literal["static_uncompressed", "dynamic_uncompressed", "static_compressed", "dynamic_compressed"] = (
        "static_compressed"
    )
    prach_format: Literal["short", "long"] = "short"
    extra: dict[str, str] = Field(default_factory=dict)

In [8]:
def make_fhi72_params(
    vlan: int,
    fronthaul_interface: str,
    ptp_interface: str,
) -> Fhi72NodeParams:
    return Fhi72NodeParams.model_validate(
        {
            "fronthaul_interface": fronthaul_interface,
            "vfs": [
                {
                    "resource_name": "ruvfiou",
                    "plane": "U",
                    "mac": "00:11:22:33:44:66",
                    "vlan": vlan,
                },
                {
                    "resource_name": "ruvfioc",
                    "plane": "C",
                    "mac": "00:11:22:33:44:67",
                    "vlan": vlan,
                },
            ],
            "reserved_cpus": "0-5",
            "irq_pattern": "ice-|ens15f[0-3]np[0-3]",
            "ptp": {"interface": ptp_interface},
        }
    )


def make_ru_config() -> dict:
    return BenetelRUParams().model_dump(mode="json")


FHI72_NODE_PARAMS = {
    "sopnode-f3": make_fhi72_params(
        vlan=801,
        fronthaul_interface="ens15f1np1",
        ptp_interface="ens15f2np2",
    ),
    "sopnode-f4": make_fhi72_params(
        vlan=802,
        fronthaul_interface="ens15f2np2",
        ptp_interface="ens15f3np3",
    ),
}

RU_TARGETS = {
    "sopnode-f3": {"name": "benetel1", "component_name": "du1", "ip": None},
    "sopnode-f4": {"name": "benetel2", "component_name": "du2", "ip": None},
}

MISCELLANEOUS = {
    "fhi72_params": {
        node_name: params.model_dump(mode="json")
        for node_name, params in FHI72_NODE_PARAMS.items()
    },
    "ru_targets": RU_TARGETS,
    "ru_config": {f"benetel{i}": make_ru_config() for i in range(1, 3)},
    "derived_fhi72_params": {},
    "du_resources": {
        "requests": {"nf": {"hugepages_1gi": "8Gi"}},
        "limits": {"nf": {"hugepages_1gi": "8Gi"}},
    },
}

for node_name, params in FHI72_NODE_PARAMS.items():
    kubelet_params = BAREMETAL_NODES[node_name]["k8s_params"]["kubelet_params"]
    kubelet_params["reserved_system_cpus"] = params.reserved_cpus
    kubelet_params["topology_manager_policy"] = params.topology_manager_policy


### Derive the node and RU configuration

This cell derives SR-IOV resource names and each RU configuration from the local models and the radio settings. The values are then passed to Ansible and the DU chart through `miscellaneous`.

In [9]:
from ifran.utils.radio import get_rbs, nr_arfcn_to_frequency_hz
from metamodel_3gpp.models.common import DuplexMode


def make_derived_fhi72_params(node_params: Fhi72NodeParams) -> dict:
    resources = [
        f"{node_params.resource_prefix}/{node_params.vf(plane).resource_name}"
        for plane in ("U", "C")
    ]
    return {
        "sriov_resources": resources,
        "pci_device_env_vars": [
            "PCIDEVICE_" + resource.upper().replace(".", "_").replace("/", "_")
            for resource in resources
        ],
    }


def make_benetel_tdd_config(radio_config) -> dict[str, str]:
    tdd = radio_config.tddConfiguration
    if radio_config.duplexMode is not DuplexMode.TDD or tdd is None:
        raise ValueError("Benetel RU configuration supports TDD only.")

    slots = round(float(tdd.periodicityMilliseconds) * int(radio_config.subcarrierSpacingKhz) / 15)
    has_special_slot = tdd.nrofDownlinkSymbols > 0 or tdd.nrofUplinkSymbols > 0
    pattern = "D" * tdd.nrofDownlinkSlots + ("S" if has_special_slot else "") + "U" * tdd.nrofUplinkSlots
    if len(pattern) != slots:
        raise ValueError(f"TDD pattern {pattern!r} does not fill its {slots}-slot period.")
    if 2 * slots > 10:
        raise ValueError(f"Benetel supports at most five slots per TDD period; got {slots}.")

    guard_symbols = 14 - tdd.nrofDownlinkSymbols - tdd.nrofUplinkSymbols
    if guard_symbols < 0:
        raise ValueError("The TDD special slot cannot contain more than 14 downlink and uplink symbols.")

    return {
        "tdd_pattern_1": pattern,
        "tdd_pattern_2": pattern,
        "special_slots_symbols": (
            "D" * tdd.nrofDownlinkSymbols
            + "G" * guard_symbols
            + "U" * tdd.nrofUplinkSymbols
        ),
    }


def make_benetel_ru_config(ru_name: str, node_name: str, component_name: str) -> dict[str, str]:
    ru_meta = metamodel.rus[ru_name]
    radio_config = metamodel.ran.components[component_name].config.radioConfiguration
    node_params = FHI72_NODE_PARAMS[node_name]
    params = BenetelRUParams.model_validate(MISCELLANEOUS["ru_config"][ru_name])

    mimo_modes = {
        (4, 4): "1_2_3_4_4x4",
        (4, 2): "1_2_3_4_4x2",
    }
    mimo_mode = mimo_modes.get((ru_meta.numberOfTxPorts, ru_meta.numberOfRxPorts))
    if mimo_mode is None:
        raise ValueError(
            f"Benetel RU {ru_name!r} does not support "
            f"{ru_meta.numberOfTxPorts}T{ru_meta.numberOfRxPorts}R."
        )

    bandwidth_rb = get_rbs(radio_config.subcarrierSpacingKhz, radio_config.channelBandwidthMhz)
    center_frequency_hz = nr_arfcn_to_frequency_hz(radio_config.pointAFrequencyArfcn) + (
        bandwidth_rb * 12 * int(radio_config.subcarrierSpacingKhz) * 1000 // 2
    )
    vlan = f"{node_params.vf('U').vlan:x}"

    config = {
        "mimo_mode": mimo_mode,
        "downlink_scaling": str(params.downlink_scaling),
        "prach_format": params.prach_format,
        "compression": params.compression,
        "c_plane_du_mac": node_params.vf("C").mac.lower(),
        "u_plane_du_mac": node_params.vf("U").mac.lower(),
        "u_plane_du_vlan_uplink": vlan,
        "u_plane_du_vlan_downlink": vlan,
        "c_plane_du_vlan": vlan,
        "bandwidth_hz": str(radio_config.channelBandwidthMhz * 1_000_000),
        "centre_frequency_hz": str(center_frequency_hz),
        "tx_power_dbm": f"{params.tx_power_dbm:.6f}",
        **make_benetel_tdd_config(radio_config),
    }
    return config | params.extra


MISCELLANEOUS["derived_fhi72_params"] = {
    node_name: make_derived_fhi72_params(params)
    for node_name, params in FHI72_NODE_PARAMS.items()
}
MISCELLANEOUS["ru_config"] = {
    target["name"]: make_benetel_ru_config(
        ru_name=target["name"],
        node_name=node_name,
        component_name=target["component_name"],
    )
    for node_name, target in RU_TARGETS.items()
}

In [10]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "provisioners": {
            "bi": {
                "type": "bi",
                "params": {
                    "project": SLICES_PROJECT,
                    "experiment": SLICES_EXPERIMENT,
                    "infra": SLICES_BI_INFRA,
                    "flavor": SLICES_BI_VM_FLAVOR,
                    "node_duration": SLICES_BI_VM_DURATION,
                    "create_experiment": False,
                },
            },
            "pos": {
                "type": "pos",
                "params": {
                    "attempt_reservation": True,
                    "image": "ubuntu-questing-rt@2025-10-29T19:34:30+00:00",
                    "is_staged_image": True,
                    "boot_params": POS_BOOT_PARAMS,
                },
            },
            "r2lab": {
                "type": "r2lab",
                "params": {
                    "attempt_reservation": False,
                    "r2lab_slice": R2LAB_SLICE,
                    "reservation_params": {
                        "email": R2LAB_EMAIL,
                        "password": R2LAB_PASSWORD,
                    },
                },
            },
        },
        "nodes": VMS | BAREMETAL_NODES,
        "ran": {
            "implementation": "oai",
            "components": {
                "cucp": {
                    "type": "cu_cp",
                    "advanced_config": {
                        "useAdditionalOptions": "--telnetsrv --telnetsrv.shrmod ci --telnetsrv.listenport 9091"
                    },
                },
                "cuup": {"type": "cu_up"},
                "du1": make_du_deployment("sopnode-f3"),
                "du2": make_du_deployment("sopnode-f4"),
            },
        },
        "rus": {
            f"benetel{i}": make_ru_deployment() for i in range(1, 3)
        },
        "ues": {
            f"qhat{i:02d}": make_ue_deployment() for i in range(1, 4)
        },
        "miscellaneous": MISCELLANEOUS,
    }
)

> The configurations are referenced from: https://github.com/sopnode/oai5g-rru/blob/main/rru/du-config-benetel.yaml

In [11]:
metamodel.to_yaml_file("input_metamodel.yaml", exclude_defaults=False)

In [12]:
deployment_model.to_yaml_file("input_deployment_model.yaml", exclude_defaults=False)

## Provision nodes

In [13]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [14]:
await prov.provision_nodes()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/pos/reserve : Select testbed 'sophia'] **********************
changed: [localhost]

TASK [provisioning/pos/reserve : Reserve 'sopnode-f3, sopnode-f4' for '120' minutes] ***
changed: [localhost]

PLAY RECAP *********************************************************************
localhost                  : ok=2    changed=2    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is emp

Use one of the sopnodes as SSH jumphost to access the Benetel RUs:

In [15]:
for ru in prov.deployment_model.rus.values():
    ru.ssh_info.jumphost = prov.deployment_model.nodes["sopnode-f3"].ssh_info

## Deploy Kubernetes

In [16]:
from ifran import K8SManagement

k8s = K8SManagement(prov)

In [17]:
await k8s.deploy()


PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

PLAY [node2] *******************************************************************

PLAY [node3] *******************************************************************

PLAY [sopnode-f4] **************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

PLAY [sopnode-f3] **************************************************************

PLAY [node4] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f4'] ***********

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f3'] ***********

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************
changed: [sopnod

## Deploy the 5G core

In [18]:
from ifran import CoreManagement

core = CoreManagement(k8s)

In [19]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

PLAY [node1] 

## Configure the FHI72 DU nodes

The host playbook applies NIC, VF, IRQ, CPU, sysctl, SR-IOV device plugin, and PTP setup. Kubelet CPU policy is supplied through each node's existing `k8s_params` during cluster deployment.

In [20]:
from ifran import MiscellaneousManagement

misc = MiscellaneousManagement(core)

In [21]:
print(misc.get_ansible_inventory())

[core_nodes]
node1 ansible_host=10.123.12.202 ansible_port=22 ansible_user=ziyad ansible_ssh_timeout=30 ansible_ssh_pipelining=true ansible_python_interpreter=/usr/bin/python3 ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'
node2 ansible_host=10.123.98.201 ansible_port=22 ansible_user=ziyad ansible_ssh_timeout=30 ansible_ssh_pipelining=true ansible_python_interpreter=/usr/bin/python3 ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.the

In [22]:
await misc.run_playbook(
    playbook="./misc/playbooks/fhi72.yaml",
    roles_paths=["./misc/roles"],
)


PLAY [Configure O-RAN 7.2 fronthaul nodes] *************************************

TASK [Gathering Facts] *********************************************************
ok: [sopnode-f3]
ok: [sopnode-f4]

TASK [fhi72 : Select this node's FHI72 parameters] *****************************
ok: [sopnode-f3]
ok: [sopnode-f4]

TASK [Verify DPDK kernel prerequisites] ****************************************
included: fhi72/prerequisites for sopnode-f3, sopnode-f4

TASK [fhi72/prerequisites : Check IOMMU groups] ********************************
ok: [sopnode-f4]
ok: [sopnode-f3]

TASK [fhi72/prerequisites : Read current 1G hugepage count] ********************
ok: [sopnode-f4]
ok: [sopnode-f3]

TASK [fhi72/prerequisites : Allocate configured 1G hugepages] ******************
skipping: [sopnode-f3]
skipping: [sopnode-f4]

TASK [fhi72/prerequisites : Read allocated 1G hugepage count] ******************
ok: [sopnode-f3]
ok: [sopnode-f4]

TASK [fhi72/prerequisites : Stop when required kernel prerequisites ar

## Provision and configure the Benetel RUs

In [23]:
from ifran import ProvisioningManagement

prov_radio = ProvisioningManagement(misc)

In [24]:
await prov_radio.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhos

In [25]:
print(misc.get_ansible_inventory())

[core_nodes]
node1 ansible_host=10.123.12.202 ansible_port=22 ansible_user=ziyad ansible_ssh_timeout=30 ansible_ssh_pipelining=true ansible_python_interpreter=/usr/bin/python3 ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'
node2 ansible_host=10.123.98.201 ansible_port=22 ansible_user=ziyad ansible_ssh_timeout=30 ansible_ssh_pipelining=true ansible_python_interpreter=/usr/bin/python3 ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.the

In [26]:
misc_ru = MiscellaneousManagement(prov_radio)

In [27]:
ru_targets = misc_ru.deployment_model.miscellaneous["ru_targets"]

for node_name, target in ru_targets.items():
    ru = misc_ru.deployment_model.rus[target["name"]]
    if not ru.ips:
        raise ValueError(f"No management IP was returned for RU {target['name']!r}.")
    target["ip"] = str(ru.ips[0])
    ru.ips = []

In [28]:
await misc_ru.run_playbook(
    playbook="./misc/playbooks/netplan.yaml",
    roles_paths=["./misc/roles"],
)


PLAY [Apply netplan on DU nodes] ***********************************************

TASK [Gathering Facts] *********************************************************
ok: [sopnode-f3]
ok: [sopnode-f4]

TASK [netplan : Upload netplan config to remote] *******************************
changed: [sopnode-f4]
changed: [sopnode-f3]

TASK [netplan : Generate netplan config] ***************************************
changed: [sopnode-f4]
changed: [sopnode-f3]

TASK [netplan : Apply netplan config] ******************************************
changed: [sopnode-f4]
changed: [sopnode-f3]

TASK [netplan : Install lldpd] *************************************************
changed: [sopnode-f3]
changed: [sopnode-f4]

TASK [netplan : Enable and start lldpd service] ********************************
ok: [sopnode-f4]
ok: [sopnode-f3]

PLAY RECAP *********************************************************************
sopnode-f3                 : ok=6    changed=4    unreachable=0    failed=0    skipped=0    rescued=

In [29]:
await misc_ru.run_playbook(
    playbook="./misc/playbooks/benetel.yaml",
    roles_paths=["./misc/roles"],
    retries=1, # do not retry many times
)


PLAY [Install Benetel RU configuration] ****************************************

TASK [benetel/configure : Render RU configuration locally] *********************
changed: [benetel1 -> localhost]
changed: [benetel2 -> localhost]

TASK [benetel/configure : Install the rendered RU configuration] ***************
changed: [benetel2]
changed: [benetel1]

TASK [benetel/configure : Save the configuration for post-reboot verification] ***
ok: [benetel1]
ok: [benetel2]

TASK [benetel/configure : Remove temporary rendered configuration] *************
ok: [benetel1 -> localhost]
changed: [benetel2 -> localhost]

PLAY [Reboot RU when configuration changed or reboot was forced] ***************

TASK [benetel/reboot : Read the current RU boot ID] ****************************
ok: [benetel1]
ok: [benetel2]

TASK [benetel/reboot : Reboot the RU] ******************************************
changed: [benetel2]
changed: [benetel1]

TASK [benetel/reboot : Allow the RU to start rebooting] ******************

## Deploy the RAN

Deploy the CU first, then the two DUs. The custom DU chart reads the node-specific SR-IOV resources and hugepages from `miscellaneous`.

In [30]:
from ifran import RANManagement

ran = RANManagement(misc_ru)

In [31]:
await ran.deploy(components=["cucp", "cuup"])


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********
changed: [node1]
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'oai-cucp' if provided on localhost] ***

TASK [helm/prepare : Compress custom Helm chart directory for 'oai-cuup' if provided on localhost] ***
skipping: [node1]
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'oai-cucp' on localhost] ***

TASK [helm/prepare : Compress built-in Helm chart directory for 'oai-cuup' on localhost] ***
changed: [node1 -> localhost]changed: [node1 -> localhost]


TASK [helm/prepare : Extract Helm chart archive for 'oai-cuup' from localhost] ***

TASK [helm/prepare : Extract Helm chart archive for 'oai-cucp

In [32]:
await ran.deploy(components=["du1"])


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'oai-du1' if provided on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Compress built-in Helm chart directory for 'oai-du1' on localhost] ***
skipping: [node1]

TASK [helm/prepare : Extract Helm chart archive for 'oai-du1' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'oai-du1' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-du1'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'oai-du1' if provided] ***
changed: [node1] => (item=/Users/zimabrou/Projects/ifran/examples/benetel/custom_du_chart/values.yaml.j2)

TASK [helm/prepare : Render built-in templated Helm f

In [33]:
print(await ran.get_logs(components=["du1"], quiet=True))

Virtual PCI devices: 0000:3a:09.0 0000:3a:09.1
Available CPUs: 6-19 (count=14)
CPU allocation: L1 RX 6, L1 TX 7, RU 8, system 9, IO 10, worker 11, thread pool 12,13,14,15,16,17,18,19

# Active DUs
Active_gNBs:
  - du1
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - {gNB_ID: 0x1, gNB_DU_ID: 0x1, gNB_name: du1, tracking_area_code: 1, plmn_list: [{mcc: 001, mnc: 01, mnc_length: 2, snssaiList: [{sst: 1}]}], nr_cellid: 1,
    ##### Physical parameters:
    "min_rxtxtime": 6, "pdsch_AntennaPorts_XP": 2, "pdsch_AntennaPorts_N1": 2, "pusch_AntennaPorts": 4, "do_CSIRS": 1, "do_SRS": "periodic", servingCellConfigCommon: [
      #spCellConfigCommon
      {physCellId: 0,
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 630048, dl_absoluteFrequencyPointA: 626772,
        #scs-SpecificCarrierList
        "dl_offstToCarrier": 0,
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz6

## Deploy the second DU

In [34]:
await ran.deploy(components=["du2"])


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du2' chart] **********
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'oai-du2' if provided on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Compress built-in Helm chart directory for 'oai-du2' on localhost] ***
skipping: [node1]

TASK [helm/prepare : Extract Helm chart archive for 'oai-du2' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'oai-du2' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-du2'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'oai-du2' if provided] ***
changed: [node1] => (item=/Users/zimabrou/Projects/ifran/examples/benetel/custom_du_chart/values.yaml.j2)

TASK [helm/prepare : Render built-in templated Helm f

In [35]:
print(await ran.get_logs(components=["du2"], quiet=True))

Virtual PCI devices: 0000:3a:11.0 0000:3a:11.1
Available CPUs: 6-19 (count=14)
CPU allocation: L1 RX 6, L1 TX 7, RU 8, system 9, IO 10, worker 11, thread pool 12,13,14,15,16,17,18,19

# Active DUs
Active_gNBs:
  - du2
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - {gNB_ID: 0x1, gNB_DU_ID: 0x2, gNB_name: du2, tracking_area_code: 1, plmn_list: [{mcc: 001, mnc: 01, mnc_length: 2, snssaiList: [{sst: 1}]}], nr_cellid: 2,
    ##### Physical parameters:
    "min_rxtxtime": 6, "pdsch_AntennaPorts_XP": 2, "pdsch_AntennaPorts_N1": 2, "pusch_AntennaPorts": 4, "do_CSIRS": 1, "do_SRS": "periodic", servingCellConfigCommon: [
      #spCellConfigCommon
      {physCellId: 1,
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 649920, dl_absoluteFrequencyPointA: 646644,
        #scs-SpecificCarrierList
        "dl_offstToCarrier": 0,
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz6

## Connect the UEs

In [36]:
from ifran import UEManagement

ue = UEManagement(ran)

In [37]:
await ue.configure()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Configure 'qhat03'] ******************************************************

TASK [Configure 'qhat02'] ******************************************************

In [38]:
await ue.connect()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Connect 'qhat01'] ********************************************************

TASK [Connect 'qhat02'] ********************************************************

In [39]:
print(await ran.get_logs(components=["du1"], quiet=True))

Virtual PCI devices: 0000:3a:09.0 0000:3a:09.1
Available CPUs: 6-19 (count=14)
CPU allocation: L1 RX 6, L1 TX 7, RU 8, system 9, IO 10, worker 11, thread pool 12,13,14,15,16,17,18,19

# Active DUs
Active_gNBs:
  - du1
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - {gNB_ID: 0x1, gNB_DU_ID: 0x1, gNB_name: du1, tracking_area_code: 1, plmn_list: [{mcc: 001, mnc: 01, mnc_length: 2, snssaiList: [{sst: 1}]}], nr_cellid: 1,
    ##### Physical parameters:
    "min_rxtxtime": 6, "pdsch_AntennaPorts_XP": 2, "pdsch_AntennaPorts_N1": 2, "pusch_AntennaPorts": 4, "do_CSIRS": 1, "do_SRS": "periodic", servingCellConfigCommon: [
      #spCellConfigCommon
      {physCellId: 0,
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 630048, dl_absoluteFrequencyPointA: 626772,
        #scs-SpecificCarrierList
        "dl_offstToCarrier": 0,
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz6

In [40]:
print(await ran.get_logs(components=["du2"], quiet=True))

Virtual PCI devices: 0000:3a:11.0 0000:3a:11.1
Available CPUs: 6-19 (count=14)
CPU allocation: L1 RX 6, L1 TX 7, RU 8, system 9, IO 10, worker 11, thread pool 12,13,14,15,16,17,18,19

# Active DUs
Active_gNBs:
  - du2
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - {gNB_ID: 0x1, gNB_DU_ID: 0x2, gNB_name: du2, tracking_area_code: 1, plmn_list: [{mcc: 001, mnc: 01, mnc_length: 2, snssaiList: [{sst: 1}]}], nr_cellid: 2,
    ##### Physical parameters:
    "min_rxtxtime": 6, "pdsch_AntennaPorts_XP": 2, "pdsch_AntennaPorts_N1": 2, "pusch_AntennaPorts": 4, "do_CSIRS": 1, "do_SRS": "periodic", servingCellConfigCommon: [
      #spCellConfigCommon
      {physCellId: 1,
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 649920, dl_absoluteFrequencyPointA: 646644,
        #scs-SpecificCarrierList
        "dl_offstToCarrier": 0,
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz6

In [41]:
ping = await prov_radio.run_ue_command(command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], quiet=True, flatten=False)

In [42]:
print(ping["qhat01"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.2 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=26.5 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=25.5 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=28.8 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=27.3 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=25.8 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=26.3 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5008ms
rtt min/avg/max/mdev = 25.521/26.733/28.838/1.099 ms


In [43]:
print(ping["qhat02"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.3 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=27.6 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=18.3 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=29.3 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=28.3 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=26.8 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=29.9 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5008ms
rtt min/avg/max/mdev = 18.318/26.726/29.942/3.897 ms


In [44]:
print(ping["qhat03"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.4 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=25.9 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=29.7 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=27.7 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=25.7 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=28.7 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=24.3 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5009ms
rtt min/avg/max/mdev = 24.322/27.010/29.687/1.857 ms


In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

## Cleanup

In [45]:
# await prov_radio.unprovision()